In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
dataset_path = "/content/drive/MyDrive/Colab Notebooks/odi_parsed.csv"
df = pd.read_csv(dataset_path)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error


def create_states(df):
    first_innings = df[df['inning'] == 1].copy()

    first_innings_totals = df[df['inning'] == 1].groupby('match_id')['cumulative_runs'].max()
    second_innings = df[df['inning'] == 2].copy()
    second_innings = second_innings.merge(first_innings_totals.rename('target_score'),
                                        left_on='match_id', right_index=True)

    # state representations
    first_innings_states = []
    first_innings_rewards = []

    for match_id, group in first_innings.groupby('match_id'):
        total_runs = group['cumulative_runs'].max()
        for _, row in group.iterrows():
            # State dimensions: [over, wickets, score_band, ball_in_over, is_legal]
            state = [
                min(row['over'], 50),  # over (0-50)
                min(row['wickets_lost'], 10),  # wickets (0-10)
                min(row['score_band'], 49),  # score band (0-49)
                min(row['ball_in_over'], 5),  # ball in over (0-5)
                1 if row['is_legal'] else 0  # is legal (0 or 1)
            ]
            first_innings_states.append(state)

            # Reward is fraction of total runs scored in this state
            # reward = row['runs_batsman'] / total_runs if total_runs > 0 else 0
            reward = ((total_runs - row['cumulative_runs']) / total_runs * 100) if total_runs > 0 else 0
            # reward = (row['cumulative_runs'] / total_runs * 100) if total_runs > 0 else 0
            first_innings_rewards.append(reward)

    second_innings_states = []
    second_innings_rewards = []

    for match_id, group in second_innings.groupby('match_id'):
        total_runs = group['cumulative_runs'].max()
        target = group['target_score'].iloc[0]
        for _, row in group.iterrows():
            # State dimensions: [over, wickets, score_band, target_band, ball_in_over, is_legal]
            state = [
                min(row['over'], 50),  # over (0-50)
                min(row['wickets_lost'], 10),  # wickets (0-10)
                min(row['score_band'], 49),  # score band (0-49)
                min(target // 10, 49),  # target band (0-49)
                min(row['ball_in_over'], 5),  # ball in over (0-5)
                1 if row['is_legal'] else 0  # is legal (0 or 1)
            ]
            second_innings_states.append(state)

            # Reward is fraction of total runs scored in this state
            # reward = row['runs_batsman'] / total_runs if total_runs > 0 else 0
            reward = ((total_runs - row['cumulative_runs']) / total_runs * 100) if total_runs > 0 else 0
            # reward = (row['cumulative_runs'] / total_runs * 100) if total_runs > 0 else 0
            second_innings_rewards.append(reward)

    return (
        np.array(first_innings_states), np.array(first_innings_rewards),
        np.array(second_innings_states), np.array(second_innings_rewards)
    )

first_states, first_rewards, second_states, second_rewards = create_states(df)

In [ ]:
from sklearn.metrics import mean_absolute_error
from tensorflow.keras import Input

# Define model builder (fixed warning)
def build_model(input_dim, hidden_layers=[64, 32]):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    for units in hidden_layers:
        model.add(Dense(units, activation='relu'))
    model.add(Dense(1, activation='linear'))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    return model

# Cross-validation function
def monte_carlo_cross_validation(states, rewards, n_splits=10, epochs=20, batch_size=256):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_errors = []
    fold_models = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(states), 1):
        X_train, X_test = states[train_idx], states[test_idx]
        y_train, y_test = rewards[train_idx], rewards[test_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model = build_model(input_dim=X_train.shape[1])
        model.fit(X_train_scaled, y_train, epochs=epochs, batch_size=batch_size, verbose=0)

        y_pred = model.predict(X_test_scaled).flatten()
        error = mean_absolute_error(y_test, y_pred)
        fold_errors.append(error)
        fold_models.append((model, scaler))

        print(f"Fold {fold}: MAE = {error:.4f}")

    print("\nCross-Validation Summary:")
    print(f"Mean MAE: {np.mean(fold_errors):.4f}")
    print(f"Std Dev: {np.std(fold_errors):.4f}")
    return fold_errors, fold_models


In [ ]:
# First innings training
print("Running cross-validation for first innings...")
first_errors, first_models = monte_carlo_cross_validation(first_states, first_rewards)

# Second innings training
print("\nRunning cross-validation for second innings...")
second_errors, second_models = monte_carlo_cross_validation(second_states, second_rewards)


Running cross-validation for first innings...
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
Fold 1: MAE = 4.9444
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Fold 2: MAE = 5.0599
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Fold 3: MAE = 5.0104
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Fold 4: MAE = 4.9313
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Fold 5: MAE = 5.0461
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Fold 6: MAE = 4.9618
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Fold 7: MAE = 4.8728
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Fold 8: MAE = 4.9738
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
Fold 9: MAE = 5.0061
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Fold 10: MAE = 4.9116

Cross-Validation Summary:
Mean MAE: 4.9718
Std Dev: 0.0565

Running cross-validation for second innings...
2206/2206 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Fold 1: MAE = 4.5723
2206/2206 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Fold 2: MAE = 4.5801
2206/2206 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Fold 3: MAE = 4.4424
2206/2206 ━━━

In [ ]:
from tqdm import tqdm

def evaluate_final_score_prediction(df, models, innings=1, min_over=20):
    errors = []
    matches = df[df['inning'] == innings].groupby('match_id')

    for match_id, group in tqdm(matches, desc=f"Innings {innings} Evaluation"):
        valid_rows = group[group['over'] >= min_over]
        if valid_rows.empty:
            continue

        # Sample one interruption point
        sample_row = valid_rows.sample(n=1).iloc[0]
        current_score = sample_row['cumulative_runs']
        final_score = group['cumulative_runs'].max()

        # Build state
        if innings == 1:
            state = [
                min(sample_row['over'], 50),
                min(sample_row['wickets_lost'], 10),
                min(sample_row['score_band'], 49),
                min(sample_row['ball_in_over'], 5),
                1 if sample_row['is_legal'] else 0
            ]
        else:
            target = df[(df['match_id'] == match_id) & (df['inning'] == 1)]['cumulative_runs'].max()
            state = [
                min(sample_row['over'], 50),
                min(sample_row['wickets_lost'], 10),
                min(sample_row['score_band'], 49),
                min(target // 10, 49),
                min(sample_row['ball_in_over'], 5),
                1 if sample_row['is_legal'] else 0
            ]

        # Use ensemble of all models to predict resources left
        r_vals = []
        for model, scaler in models:
            scaled_state = scaler.transform([state])
            r_left = model.predict(scaled_state)[0][0] / 100
            r_vals.append(r_left)

        r_left = np.mean(r_vals)
        r_left = np.clip(r_left, 0.05, 0.95)
        progress = 1 - r_left                       # convert to resources used

        predicted_score = np.ceil(current_score / progress)
        # predicted_score = min(predicted_score, final_score * 1.5)  # cap runaway predictions

        percent_error = abs(predicted_score - final_score) / final_score * 100
        errors.append(percent_error)

        if percent_error > 20:
            print(f"[⚠️ Error > 20%] Match: {match_id}, Over: {sample_row['over']}, r_left: {r_left:.4f}")
            print(f"  Score: {current_score}, Final: {final_score}, Pred: {predicted_score}, Error: {percent_error:.2f}%")

    print(f"\nEvaluation on {len(errors)} games after over {min_over}:")
    print(f"Mean % Error: {np.mean(errors):.2f}%, Std: {np.std(errors):.2f}%")
    return errors


In [ ]:
# Evaluate again
first_eval_errors = evaluate_final_score_prediction(df, first_models, innings=1)
second_eval_errors = evaluate_final_score_prediction(df, second_models, innings=2)


Innings 1 Evaluation:   0%|          | 0/2928 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step


Innings 1 Evaluation:   0%|          | 1/2928 [00:01<1:21:22,  1.67s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step


Innings 1 Evaluation:   0%|          | 2/2928 [00:02<53:04,  1.09s/it]  

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step


Innings 2 Evaluation: 100%|█████████▉| 2852/2853 [58:32<00:01,  1.34s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step


Innings 2 Evaluation: 100%|██████████| 2853/2853 [58:33<00:00,  1.23s/it]


Evaluation on 2692 games after over 20:
Mean % Error: 9.30%, Std: 12.71%
